# SchwarMAX — the AIC pipeline on the mock N-body galaxy: double-exponential disc

**This is the exponential-disc variant of `SchwarMAX_Nbody_AIC.ipynb`.** Two things differ:

1. **The disc** is NGC 1512's double exponential, $\rho\propto e^{-R/R_d}\,e^{-|z|/h_z}$, in
   place of the Miyamoto–Nagai disc. Its potential is solved from the density by the CylSpline
   (`get_phi_m`) with NGC 1512's settings. The bar, the bulge, the halo, the ties
   ($a_{\rm bar}=L_{\rm bar}/5$, $b_{\rm bar}=h_z$, bulge mass $=M_{\rm bar}$), the stellar mass
   tied to the light and the 12 parameters are unchanged: $\log R_d$ and $\log h_z$ take the slots
   of $\log R_s$ and $\log H_s$.
2. **The sampler** is emcee (differential-evolution + snooker moves, vectorised), NGC 1512's
   cell, in place of BlackJAX.

The Miyamoto–Nagai chain recovered a disc that was too compact ($R_s$ 5.3 against 12 kpc) and a
bar that was too short and too weak. This notebook runs the same experiment with a disc of NGC
1512's form. Every file it writes carries `RUN_TAG = 'aic_exp'`.

---

The same mock as `example_schwarmax.ipynb`. The structure of this notebook
is deliberately identical, so the two can be read side by side; the pipeline underneath changes,
and so does **one thing about the parameters: the stellar mass is tied to the light** (see
"One stellar normalisation" below).

| | `example_schwarmax.ipynb` | this notebook |
|---|---|---|
| objective | log-mean-exp over data bootstraps of $-\tfrac12\sum\chi^2$ | $\log L=-\tfrac12(\chi^2_{\rm kin}+2m_{\rm eff,kin}+\chi^2_{\rm den})/S^2$ |
| bootstraps | marginalised over | spent on $m_{\rm eff}$; the fit itself is unperturbed |
| model builder | `build_model` | `build_model_acc_meff_fourier` |
| 3-D block | regular $(R,z,\phi)$ cells | Fourier rings, even $m\le6$ |
| the $\chi^2$ scale | jackknife $\Delta\chi^2$ | **$\sqrt{2N_{\rm kin}}$, validated by $\sigma_{\rm AIC}$** (with $\sigma_{\Delta\rm AIC}$ measured alongside) |
| stellar mass | $\log M_{\rm disc}$, $\log M_{\rm bar}$ **and** $M/L$, all free | **derived**: $M/L\times L_{\rm box}/p_{\rm box}(\theta)$, split by $f_{\rm bar}$ |
| parameters | 12 | 12 — $f_{\rm bar}$ replaces the two masses; $\log_{10}\lambda$ is sampled |
| stellar disc | Miyamoto–Nagai | **double exponential, CylSpline potential** |
| sampler | BlackJAX adaptive RMH | **emcee** (affine-invariant ensemble, NGC 1512's cell) |
| tracer positions | `load_data_bootstrap` | **unchanged** |

$m_{\rm eff}$ is the effective number of free parameters the orbit solution actually uses
(Thomas & Lipka 2022, eq. 15): noise is added to the FIT, the fit is repeated at the same
$\lambda$, and the response is measured. Judging orbit models by $\chi^2$ alone rewards
flexibility, and the parameter it corrupts most is the inclination — which this mock knows the
truth of, so this is a test with an answer.

> **Read cell 6 before running.** The mock must be flipped in handedness for this pipeline.
> `get_w0_new` (used by `build_model`) seeds clockwise; `get_w0_new_improved` (used by every
> `build_model_acc*`) seeds anticlockwise. Measured both ways, the two pipelines swap places.

# Import functions

In [ ]:
# ===========================================================================
# XLA MEMORY FLAGS MUST PRECEDE THE FIRST jax IMPORT -- jax reads them once, at backend init.
# ===========================================================================
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]  = "true"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

# Mount FIRST: it needs a consent click and is slow, so it should not wait on the install.
from google.colab import drive
drive.mount('/content/drive')

# ===========================================================================
# PIN jax 0.7.2 -- and pin ALL FOUR packages, not two.
# The original line `!pip install blackjax jax==0.7.2 jaxlib==0.7.2` pinned only jax and jaxlib,
# leaving jax-cuda12-plugin / jax-cuda12-pjrt at the newer version -> XlaRuntimeError
# "Unexpected PJRT_ExecuteOptions size" on the first jitted call, which surfaces at
# `from utils import ...` and looks like an import error but is not.
# `jax[cuda12]==0.7.2` pins the matched set.
#
# The pin is required because schwarmax/model.py uses jnp.clip(a_min=, a_max=), which newer jax
# removed (6 calls).  Nothing above this point imports jax, so on a fresh runtime the pin takes
# effect immediately and NO RESTART is needed -- jaxlib is a compiled .so that CPython cannot
# unload, so a downgrade cannot take effect in a process that already holds it.
# ===========================================================================
TARGET = '0.7.2'
import sys, subprocess, shutil, importlib.metadata as _md
_already = ('jax' in sys.modules) or ('jaxlib' in sys.modules)
def _v(p_):
    try: return _md.version(p_)
    except Exception: return None
_pkgs = ('jax', 'jaxlib', 'jax-cuda12-plugin', 'jax-cuda12-pjrt',
         'jax-cuda13-plugin', 'jax-cuda13-pjrt')
print('jax already imported in this session: %s | installed: %s'
      % (_already, {p_: _v(p_) for p_ in _pkgs}))

# ---------------------------------------------------------------------------------------
# REMOVE MISMATCHED CUDA PLUGINS *BEFORE* ANY jax IMPORT.      (added 2026-10-04)
# Colab's image ships whatever CUDA plugin matches ITS jax, not ours.  On 2026-10-04 that
# became jax-cuda13-* 0.11.1.  jax 0.7.2 discovers it, the plugin REGISTERS the PJRT API for
# device type "cuda" and only then crashes; the matched cuda12 plugin is refused with
# ALREADY_EXISTS, and the first jitted call dies with
#     Unexpected PJRT_ExecuteOptions size: expected 112, got 88
# i.e. the framework is 0.7.2 but the executing plugin is 0.11.1.  It looks like a code bug.
# Pinning jax/jaxlib/cuda12 is NOT enough: any OTHER jax-cuda* at a different version has to
# go.  Written generically so a future cuda14 image is handled without another patch.
# ---------------------------------------------------------------------------------------
def _stray_cuda_plugins():
    out = set()
    for _d in _md.distributions():
        _n = (_d.metadata['Name'] or '')
        if _n.lower().startswith('jax-cuda') and _d.version != TARGET:
            out.add(_n)
    return sorted(out)

_stray = _stray_cuda_plugins()
if _stray:
    if _already:
        raise SystemExit('Mismatched CUDA plugins %s are loaded and jax is ALREADY imported. '
                         'Runtime > Restart session, then re-run this cell FIRST.' % _stray)
    print('removing mismatched CUDA plugins (they would hijack the "cuda" device): %s' % _stray)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', *_stray],
                   capture_output=True, text=True)
    _left = _stray_cuda_plugins()
    assert not _left, ('could not remove %s -- uninstall by hand and RESTART the runtime' % _left)
    # a leftover jax_plugins/xla_cudaNN directory is still DISCOVERED even with the
    # distribution gone, so check the namespace package too
    import glob as _glob
    for _sp in sys.path:
        for _p in _glob.glob(os.path.join(_sp, 'jax_plugins', 'xla_cuda*')):
            if not _p.endswith('xla_cuda12'):
                print('   removing leftover plugin directory', _p)
                shutil.rmtree(_p, ignore_errors=True)

def _verify():
    import jax, jax.numpy as jnp
    print('\njax %s | jaxlib %s | plugin %s' % (jax.__version__, _v('jaxlib'), _v('jax-cuda12-plugin')))
    print('devices:', jax.devices())
    if jax.__version__ != TARGET:
        raise SystemExit('jax is %s, not %s -- Runtime > Restart session, then re-run this cell.'
                         % (jax.__version__, TARGET))
    if not any(d.platform == 'gpu' for d in jax.devices()):
        raise SystemExit('No GPU -- Runtime > Change runtime type > GPU, then restart.')
    jnp.clip(jnp.asarray(99.0), a_min=1, a_max=15)     # the call that needs 0.7.2
    print('float32 (x64 disabled):', not jax.config.jax_enable_x64)
    print('\nSTACK PINNED AT %s AND WORKING -- continue.' % TARGET)

if _v('jax') == TARGET and _v('jaxlib') == TARGET:
    _verify()
else:
    print('\ninstalling the matched %s stack (~2 min) ...' % TARGET)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'jax[cuda12]==%s' % TARGET],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise SystemExit('pip could not install the %s stack on Python %s.' % (TARGET, sys.version.split()[0]))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                    'blackjax', 'fastprogress', 'jaxopt', 'typing-extensions', 'corner', 'emcee'], check=False)
    if _already:
        print('\njax was ALREADY imported -- Runtime > Restart session, then re-run this cell.')
    else:
        _verify()

import os, sys
# This folder IS the package: model.py, utils.py ... sit at its top level, next to example/.
# Upload the whole folder to My Drive and the first candidate below is found automatically.
# Edit DRIVE_FOLDER if you renamed it; or put this notebook beside model.py and run it locally.
DRIVE_FOLDER = '/content/drive/MyDrive/SchwarMAX-Nbody'
path = None
for _c in (DRIVE_FOLDER, '/content/drive/MyDrive/SchwarMAX-Nbody',
           '/content/drive/MyDrive/SchwarMAX', os.getcwd(), os.path.dirname(os.getcwd())):
    _c = _c.rstrip('/') + '/'
    if os.path.isfile(_c + 'model.py') and os.path.isdir(_c + 'example'):
        path = _c
        break
assert path is not None, (
    'could not find the SchwarMAX folder.  Set DRIVE_FOLDER to the uploaded folder -- the one '
    'that contains model.py and example/.  Looked in: %s' % DRIVE_FOLDER)
print('SchwarMAX root:', path)
sys.path.insert(0, path)

import jax, jax.numpy as jnp
import numpy as np, pandas as pd, pickle, time, corner
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from functools import partial

from utils import logMenc_logc_to_logM_logRs, load_data_bootstrap, makeRotationMatrix
from constants import EPSILON, KPCGYR_TO_KMS
from model import build_model_acc_meff_fourier
from potentials import NFW_potential, T3_density, T3_potential, V4_density, V4_potential
from CylindricalSpline import get_phi_m, evaluate_phi, evaluate_phi_axisymmetric, get_acc

# The Fourier 3-D block and m_eff are recent additions; fail loudly if the uploaded schwarmax
# predates them rather than halfway through a 12-hour chain.
import model as _m
_need = ('fourier3d', 'meff_fourier', 'sigma_scheme', 'pow_spacing')
_miss = [f for f in _need if f not in getattr(_m, 'SCHWARMAX_FEATURES', ())]
assert not _miss, ('the SchwarMAX on Drive is OLDER than this notebook: missing %s. '
                   'Re-upload the package to %s' % (_miss, path))
print(jax.devices(), '| x64 =', jax.config.jax_enable_x64)


# Use the Schwarzschild function in SchwarMAX to build model likelihood

The stellar density is a **double-exponential disc** (NGC 1512's form) plus the parent's T3 bar
and V4 bulge; the halo is the parent's NFW. The bar, bulge and halo potentials are analytic. The
disc potential has no closed form, so it is **solved from the density by the CylSpline**,
`get_phi_m`, at every $\theta$, with the settings of the NGC 1512 notebooks (this builder reads
them out of that notebook's own call).

`build_model_acc_meff_fourier` takes five callables: the density, the potential, the
acceleration, and the $m=0$ potential and density for the Jeans seeding. They depend on the disc
spline of the current $\theta$, so `make_model_funcs` builds all five **inside** `run_model`, as
NGC 1512 does. The disc acceleration is the spline's analytic gradient (`get_acc`); the analytic
part is $-\nabla\Phi$ by autodiff, as in the parent.

**Accuracy of the disc potential**, measured against AGAMA's exact `Disk` potential (float32, as
on the GPU, $R_d=6.47$ kpc):

| $h_z$ | disc $v_c$, max error | $a_z(R=4)$ at $z=h_z/2$ |
|---|---|---|
| 0.75 kpc (the truth) | 3.6% (−2.2 km/s at 3 kpc) | 6% |
| 0.25 kpc | 8.7% | 86% |
| 0.75 kpc, `CYL_NINT = 40000` | 1.0% | 0.0% |
| 0.25 kpc, `CYL_NINT = 40000` | 2.4% | 0.3% |

The quadrature (`N_int`), not the grid, sets the error. The NGC 1512 value is kept; raising
`CYL_NINT` in cell 4 is the knob if the chain wanders to thin discs.

In [ ]:
# ---- Fixed V4 bulge and bar parameters (as in example_schwarmax.ipynb) ----
V4_A, V4_B, V4_L, V4_GAMMA = 0.5, 0.5, 0.1, 0.0
GAMMA_BAR = 1.0


# ---- the DOUBLE-EXPONENTIAL disc: NGC 1512's rho_disc, in place of the Miyamoto-Nagai disc ----
def rho_expdisc(x, y, z, p):
    """M / (4 pi Rd^2 hz) exp(-R/Rd) exp(-|z|/hz), with p = {'logM', 'Rd', 'hz'}."""
    R = jnp.sqrt(x**2 + y**2)
    return (10**p['logM']/(4*jnp.pi*p['Rd']**2*p['hz']))*jnp.exp(-R/p['Rd'])*jnp.exp(-jnp.abs(z)/p['hz'])


# Its potential is solved by the CylSpline with NGC 1512's settings, which that notebook
# benchmarked against an exact Hankel solution:
#     get_phi_m(rho, p, NR, NZ, Rmin, Rmax, Zmin, Zmax, Mmax, Nphi, N_int)
# Mmax = 0: the disc is axisymmetric.  DO NOT LOWER N_int -- the quadrature is base x base with
# base = max(9, int(sqrt(N_int))), so every N_int below 81 collapses to the same 9x9 grid.
# Measured against AGAMA's 'Disk' in float32 (Rd 6.47 kpc), max error in the disc's own force:
#     hz 0.75 (the truth)  v_c 3.6% (-2.2 km/s at 3 kpc)   a_z(R=4, z=hz/2) 6%
#     hz 0.25              v_c 8.7%                        a_z(R=4, z=hz/2) 86%
#     CYL_NINT = 40000     1.0% / 0.0% at hz 0.75;  2.4% / 0.3% at hz 0.25  (more memory)
CYL_NR, CYL_NZ = 50, 30
CYL_RMIN, CYL_RMAX, CYL_ZMIN, CYL_ZMAX = 1e-3, 50., 1e-3, 20.
CYL_MMAX, CYL_NPHI = 0, 1
CYL_NINT = 10000       # <<< the accuracy knob: see the table above


def disc_spline(p_disc):
    """The disc's CylSpline potential (a dict of spline coefficients) for p = {'logM', 'Rd', 'hz'}."""
    return get_phi_m(rho_expdisc, p_disc, CYL_NR, CYL_NZ, CYL_RMIN, CYL_RMAX,
                     CYL_ZMIN, CYL_ZMAX, CYL_MMAX, CYL_NPHI, CYL_NINT)


# SHAPE SAFETY.  The Fourier target quadrature evaluates the density on a 3-D broadcast grid,
# and `potentials._shift` (inside NFW_potential) builds jnp.array([x-x0, ...]) and needs three
# IDENTICALLY shaped arrays, while `_rotate` does `R @ vec` with R of shape (3,3).  Broadcasting
# AND flattening fixes both and changes no physics.
def _flat(f):
    @jax.jit
    def g(x, y, z, *a):
        x, y, z = jnp.broadcast_arrays(jnp.asarray(x), jnp.asarray(y), jnp.asarray(z))
        return jnp.reshape(f(x.ravel(), y.ravel(), z.ravel(), *a), x.shape)
    return g


# ================================================================= the m=0 tabulation
# The Jeans seeding needs the AXISYMMETRIC (m=0) part of the density and potential.  The disc
# spline is axisymmetric (Mmax = 0), the V4 bulge nearly so and the NFW halo exactly.
#
# TRAP.  get_w0_new_improved DIFFERENTIATES through these, so computing the bar's m=0 part as an
# azimuthal quadrature here puts n_phi copies of the integrand in the reverse-mode tape:
# MEASURED, 400 orbits, 0.98 GB -> 5.60 GB and 5.1 s -> 17.5 s.  The bar's m=0 part is
# therefore TABULATED ONCE on an (R,|z|) grid, outside the tape, and looked up by linear
# interpolation.
#
# SECOND TRAP.  The V4 bulge is deliberately NOT tabulated.  It has a = b = 0.5 kpc against a
# grid step dR = 0.48 kpc, so the grid cannot resolve it -- the same mistake, on NGC 1512's
# NSD, drove the surface-density chi2 from 13.3 to 140.8.  It also needs no averaging: it
# varies only 2-6% in azimuth, against 106-479% for the T3 bar.
M0_NR, M0_NZ, M0_RMAX, M0_ZMAX = 64, 33, 30.0, 10.0
M0_NPHI = 16          # only the BAR is averaged, and its m=0 converges fast; 16 >> 2*mmax


def make_model_funcs(params_baryon, params_halo):
    """The five callables build_model_acc_meff_fourier takes, at ONE theta:
        (density, potential, acceleration, potential_axi, density_axi)

    Called inside run_model, so they close over THIS theta's disc spline and bar table.  The
    params dicts the builder later passes back to them are accepted and ignored: model.py
    rebuilds params_baryon with the Miyamoto-Nagai key names and would drop Rd and hz.
    Safe inside a jitted function -- the closures capture tracers (NGC 1512 does the same).
    """
    p_disc = {'logM': params_baryon['logM_disc'], 'Rd': params_baryon['Rd_disc'],
              'hz': params_baryon['hz_disc']}
    dphi = disc_spline(p_disc)
    M_bar = 10.0 ** params_baryon['logM_bar']
    BAR = (M_bar, params_baryon['a_bar'], params_baryon['b_bar'], params_baryon['L_bar'], GAMMA_BAR)
    BUL = (M_bar, V4_A, V4_B, V4_L, V4_GAMMA)          # the bulge carries M_bar, as in the parent

    def density(x, y, z, pb):
        return rho_expdisc(x, y, z, p_disc) + T3_density(x, y, z, *BAR) + V4_density(x, y, z, *BUL)

    def _analytic(x, y, z, ph):                         # bar + bulge + halo
        return NFW_potential(x, y, z, ph) + T3_potential(x, y, z, *BAR) + V4_potential(x, y, z, *BUL)

    def potential(x, y, z, pb, ph):
        return evaluate_phi(x, y, z, dphi) + _analytic(x, y, z, ph)

    def acceleration(x, y, z, pb, ph):
        # disc: the spline's analytic gradient; the rest: -grad by autodiff, as in the parent
        g = jax.grad(lambda q: _analytic(q[0], q[1], q[2], ph))(jnp.array([x, y, z]))
        return get_acc(x, y, z, dphi) - g

    # the bar's m=0 part, tabulated once (TRAP above)
    Rt = jnp.linspace(0.0, M0_RMAX, M0_NR)
    zt = jnp.linspace(0.0, M0_ZMAX, M0_NZ)
    at = jnp.linspace(0.0, 2 * jnp.pi, M0_NPHI, endpoint=False)
    RR, ZZ = jnp.meshgrid(Rt, zt, indexing='ij')
    rho_tab = jnp.mean(jax.vmap(lambda a: T3_density(RR * jnp.cos(a), RR * jnp.sin(a), ZZ, *BAR))(at), 0)
    pot_tab = jnp.mean(jax.vmap(lambda a: T3_potential(RR * jnp.cos(a), RR * jnp.sin(a), ZZ, *BAR))(at), 0)
    dR, dz = M0_RMAX / (M0_NR - 1), M0_ZMAX / (M0_NZ - 1)

    def look(T, R, z):
        fR = jnp.clip(jnp.asarray(R) / dR, 0.0, M0_NR - 1.0)
        fz = jnp.clip(jnp.abs(jnp.asarray(z)) / dz, 0.0, M0_NZ - 1.0)
        return jax.scipy.ndimage.map_coordinates(
            T, [fR.ravel(), fz.ravel()], order=1, mode='nearest').reshape(jnp.shape(fR))

    def density_axi(x, y, z, pb):
        R = jnp.sqrt(x ** 2 + y ** 2)
        return rho_expdisc(x, y, z, p_disc) + look(rho_tab, R, z) + V4_density(x, y, z, *BUL)

    def potential_axi(x, y, z, pb, ph):
        R = jnp.sqrt(x ** 2 + y ** 2)
        return (NFW_potential(x, y, z, ph) + evaluate_phi_axisymmetric(x, y, z, dphi)
                + look(pot_tab, R, z) + V4_potential(x, y, z, *BUL))

    return _flat(density), _flat(potential), acceleration, potential_axi, density_axi


# Load data

The tracer particle POSITIONS come from `load_data_bootstrap` and nothing else — exactly as in
`example_schwarmax.ipynb`. They are drawn once from $p(R)\propto R\,e^{-R/3\,{\rm kpc}}$
(truncated at 12 kpc) and $p(z)\propto e^{-z/1.2\,{\rm kpc}}$, $z\ge0$, and are **fixed for the
whole MCMC** — the NNLS weights are only comparable between samples if the library does not
move. The velocities are then set by the Jeans solver at each seed's own $(R,z)$.

## The handedness flip — read this

`build_model` seeds the library **clockwise** (`getCartesianFromCylindrical_clockwise`,
$\langle v_\phi\rangle=-125$ km/s here, 14% of orbits with $L_z>0$). Every `build_model_acc*`
seeds **anticlockwise** ($+128$ km/s, 87% with $L_z>0$) — deliberately, it is the convention of
the validated NGC 4371 runs, paired with $+\Omega_{\rm bar}$ so the bar co-rotates.

This mock was generated for the first convention: it has ${\rm corr}(X,v_0)=+0.93$, negative
$v_0$ on the left and positive on the right. Run the new pipeline on it unflipped and the
library counter-rotates against the data. Measured, same $\theta$, 7500 orbits:

| mock | old pipeline | this pipeline |
|---|---|---|
| as delivered | **14 940** | 62 599 |
| flipped | 93 522 | **10 187** |

So the flip below is not optional **for this mock**. It is the substitution
$v_{\rm los}\to-v_{\rm los}$, which negates every quantity that is **odd** in $v$ and nothing
else: $v_0$, $V$, $h_1$, $h_3$. Even in $v$, and therefore untouched: $s$, $\sigma$, $h_2$,
$h_4$. Untouched for other reasons: the uncertainties (magnitudes), the stored standard normals
(symmetric), and the grid geometry and tracer positions. The cell classifies **all 47 keys** the
data dict carries and asserts the classification is complete, so nothing is left behind and a
future key cannot slip through unnoticed.

(In this mock $h_1=h_2=0$ identically — pPXF defines the Gauss-Hermite moments about $v_0$ — so
all the rotation information lives in $v_0$.)

**`FLIP_HANDEDNESS` is a manual switch and there is no automatic check.** Deciding the handedness
from fit quality is not stable enough to automate: $\rm corr(V_{\rm model},V_{\rm data})$ stays
**positive either way** (measured $+0.996$ right, $+0.893$ wrong), because the NNLS can build a
rotation field out of a counter-rotating library by up-weighting the minority of co-rotating
orbits. Set it from how your data were constructed, not from the output.

In [ ]:
filename = '/example/mock_Nbody_galaxy_beta65_gamma45_D50.pkl'

N_ORBIT = 7_500
dict_data = load_data_bootstrap(path, filename, n_samples=N_ORBIT)
NUM_VBIN = int(dict_data['total_bins'])

# ---------------------------------------------------------------- the handedness flip
# Set FLIP_HANDEDNESS = False only if your data already rotates anticlockwise on the sky.
# A quick way to tell: project a co-rotating test disc through your own geometry and compare
# the SIGN of v_los against the data.
FLIP_HANDEDNESS = True

# Every key the dict carries is classified, so the flip is complete by construction rather than
# by a hand-kept list: v_los -> -v_los negates everything ODD in v and nothing else.  The assert
# fails if a future load_data_bootstrap adds a key this cell has not been told about, instead of
# silently leaving it unflipped.
_ODD   = ('v0', 'V_data', 'h1_data', 'h3_data')            # <- negated
_EVEN  = ('s', 'sigma_data', 'h2_data', 'h4_data')         # even in v: unchanged
_MAG   = tuple(k for k in dict_data if k.endswith('_err'))              # uncertainties: positive
_NOISE = tuple(k for k in dict_data if k.endswith('_standard_normal'))  # N(0,1): symmetric
_GEOM  = ('XY_density_data', 'num_per_bin', 'bin_mapping', 'total_bins',
          'R_grid', 'z_grid', 'phi_grid', 'R_minmax', 'z_minmax', 'phi_minmax',
          'Rzphi_n_tot', 'Rzphi_n_grid', 'dR', 'dz', 'dphi', 'sample_for_integration',
          'X_regular_grid', 'Y_regular_grid', 'dX', 'dY', 'sample_for_integration_XY',
          'X_minmax', 'Y_minmax', 'nX_nY', 'w0')          # geometry, counts, tracer POSITIONS
_unclassified = set(dict_data) - set(_ODD + _EVEN + _MAG + _NOISE + _GEOM)
assert not _unclassified, ('cell 6 does not know whether these are odd or even in v: %s'
                           % sorted(_unclassified))

if FLIP_HANDEDNESS:
    for _k in _ODD:
        dict_data[_k] = -dict_data[_k]
    print('handedness FLIPPED: %s negated' % ', '.join(_ODD))
    print('                    %d other keys unchanged (%d even in v, %d uncertainties, '
          '%d noise draws, %d geometry/positions)'
          % (len(_EVEN) + len(_MAG) + len(_NOISE) + len(_GEOM),
             len(_EVEN), len(_MAG), len(_NOISE), len(_GEOM)))
else:
    print('handedness NOT flipped (FLIP_HANDEDNESS = False)')

_R = np.hypot(np.asarray(dict_data['w0'])[:, 0], np.asarray(dict_data['w0'])[:, 1])
print('mock: %d Voronoi bins | %d orbits, <R> = %.2f kpc, median %.2f, R<2 kpc frac %.3f'
      % (NUM_VBIN, N_ORBIT, _R.mean(), np.median(_R), (_R < 2).mean()))


# The AIC likelihood

Every constant that is a CHOICE is set here and marked, so the notebook states its own
configuration instead of hiding it inside a call.

## One stellar normalisation

The stellar mass is **not** sampled. As for a real galaxy, it follows from $M/L$ and the light:
$$M_\star\,p_{\rm box}(\theta)=(M/L)\,L_{\rm box},\qquad
M_{\rm disc}=(1-f_{\rm bar})M_\star,\qquad M_{\rm bar}=\tfrac12 f_{\rm bar}M_\star,$$
the half because the bar and the V4 bulge each carry $M_{\rm bar}$ (cell 4).

* $L_{\rm box}$ is the light inside a sky box $|X|,|Y|<20$ kpc centred on the galaxy, summed
  from the full image. The IFU field holds only 83.5% of the mock's light, so the full image is
  the N-body projected exactly as the mock was (`make_mock_box_light.py`, which first checks
  that it reproduces the mock's surface density). The box holds 99.96% of the light.
* $p_{\rm box}(\theta)$ is the fraction of the **model's** stellar mass that projects into the
  same box at the current shapes and viewing angles, recomputed at every $\theta$. It is not 1:
  at the generating parameters 93% of $M_{\rm disc}$ projects inside the box (the parent's
  Miyamoto–Nagai disc, with its long tail, managed 63%), while every N-body star lies within
  $R=20$ kpc.

**Why.** With $\log M_{\rm disc}$, $\log M_{\rm bar}$ and $M/L$ all free, $M/L$ scaled only
the 2-D surface-density target, while the two masses set the potential and the 3-D target.
Lowering $M/L$ made the two targets disagree, the NNLS dropped the 3-D block (whose $\chi^2$ is
not in $\log L$), and the orbits fitted the 2-D target and the kinematics better without it. The
13-parameter chain landed at $\log M/L=-0.34$ against a truth of 0 (pull $-2.0$), with the
orbits carrying 42% of the 3-D target's mass, while the potential's own stellar mass was right.
Tied to the light, $M/L$ moves the potential as well, so the kinematics measure it.

In [ ]:
# ================================================================= CONFIGURATION
# --- the 3-D Fourier block --------------------------------------------------------------
F3D_MMAX   = 6            # even harmonics only: the 4-fold symmetry makes odd m identically 0
F3D_R      = (0.2, 10.0, 15)   # R_lo, R_hi [kpc], n rings            <<< TUNE ME
F3D_Z      = (0.05, 5.0, 6)    # |z|_lo, |z|_hi [kpc], n floors       <<< TUNE ME
F3D_POW_R  = 0.5          # rings uniform in R**0.5.  NOT log: log gives the centre 53% of the
F3D_POW_Z  = 0.5          # constraint budget for 25% of the mass
F3D_LOG    = False
F3D_CLOSE_INNER = True    # bin 0 runs 0 -> e_1 in BOTH R and |z|, so no unconstrained midplane
F3D_NPHI_T = 32           # phi quadrature for the target; MUST exceed 2*mmax (Nyquist)
F3D_NSUB_T = 8            # n_sub x n_sub midpoint rule per (R,z) cell; converges as n_sub^-2
F3D_FLOOR  = 0.06
F3D_SCHEME = 'a0'         # sigma must NOT depend on the mode amplitudes -- see below
F3D_NMODE  = 1 + 2 * (F3D_MMAX // 2)
RZ_TOT     = F3D_R[2] * F3D_Z[2] * F3D_NMODE

# --- the density errors -----------------------------------------------------------------
# TWO different numbers, deliberately.  ERR_DEN_NNLS is what the NNLS SOLVES at -- it is an
# up-weighting of the density block, not a claim about the error.  ERR_DEN_LOGL is what chi2 is
# SCORED at.  Mixing them silently rescales chi2_logSig by (5/1)^2.
ERR_DEN_NNLS = 0.01       # READ-ONLY here: hard-coded at utils.py:619 as 0.01*surface_density
ERR_DEN_LOGL = 0.01       # <<< TUNE ME.  The mock carries 1%; NGC 1512 production uses 5%.
# And a THIRD number, for m_eff only.  err_den_meff = 0.0 makes the surface density a HARD
# CONSTRAINT in the flexibility count: its mock is the fit itself with no noise added, so the
# channel is charged no m_eff -- while still being scored in chi2_den.  This is Lipka & Thomas
# (2021, eq. 3)'s structure, and it is the configuration MEASURED to be best conditioned
# (NGC 1512 S/N 230: rho 0.872 -> 0.957, sigma_dAIC 43.8 -> 23.7, sigma_AIC exactly on its
# Gaussian floor).  Do not set it to ERR_DEN_LOGL without re-running that calibration.
ERR_DEN_MEFF = 0.0
# The 3-D errors are sigma(A_0) = SIG_FRAC_A0 * A_0 for the ring mass and
# sigma(c_m) = sigma(s_m) = SIG_FRAC_An * A_0 for the harmonics -- both tied to A_0, the RING
# MASS, NOT to |A_m|.  A tolerance proportional to |A_m| shrinks wherever the model is more
# axisymmetric, so the likelihood would systematically reward a stronger bar.
SIG_FRAC_A0 = 0.02        # <<< TUNE ME
SIG_FRAC_An = 0.04        # <<< TUNE ME

# --- the orbit library ------------------------------------------------------------------
N_REALIZATIONS = 4        # jitter realisations averaged into the design matrix
ANISO_B        = 1.0      # sigma_R^2 = ANISO_B * sigma_z^2.  1.0 = isotropic, matching the
                          # old pipeline's get_w0_new.  NGC 1512 production intends 1.5.
N_BOOT_MEFF    = 9        # m_eff bootstraps.  ~10 is enough: only dm_eff/dlambda must be good.
# (jeans_z_int_max is NOT a knob on this path: build_model_acc_meff_fourier asserts it is 5.0
#  and does not forward it, because get_w0_new_improved declares z_int_max static.)

# --- the temperature --------------------------------------------------------------------
# S2 divides the WHOLE bracket.  sqrt(2*N_kin) is the single-model Gaussian floor and is the
# PROVISIONAL value; the calibration cell below measures sigma_dAIC and may adopt that instead.
N_BIN_FIT = NUM_VBIN              # this mock has no inner mask
N_KIN     = 4 * N_BIN_FIT
S2        = float(np.sqrt(2.0 * N_KIN))
print('fitted bins %d -> N_kin %d -> S2 = sqrt(2 N_kin) = %.2f  (provisional)'
      % (N_BIN_FIT, N_KIN, S2))
print('3-D block: %d rings x %d modes = %d constraints' % (F3D_R[2]*F3D_Z[2], F3D_NMODE, RZ_TOT))

# --- the output files -------------------------------------------------------------------
# Every file this notebook writes carries RUN_TAG, so neither the Miyamoto-Nagai chain
# (*_aic_tied*) nor the 13-parameter one (*_aic*) is ever resumed into, or overwritten by, this one.
RUN_TAG = 'aic_exp'

# ================================================================= ONE stellar normalisation
# The stellar mass is DERIVED (markdown above):  M_star * p_box(theta) = M/L * L_BOX.
BOX_HALF = 20.0           # kpc: the sky box |X|, |Y| < BOX_HALF, centred on the galaxy
L_BOX    = 6.4078e10      # Msun at M/L = 1: the light inside it, 99.96% of the total
                          # (make_mock_box_light.py; for a real galaxy, sum the image)
BOX_NXY, BOX_NLOS, BOX_SMAX = 128, 256, 200.0   # the exponential disc: within 4.4e-4 of a
                          # 512 x 2048, s_max 400 reference for hz 0.1-2 kpc, i 40-80 deg (measured)


def _sinh_nodes(half, a, n):
    """Midpoint nodes of x = half*sinh(a*u)/sinh(a), u uniform in (-1, 1), and their dx: dense
    at the centre (the V4 bulge has a = 0.5 kpc), coarse in the smooth outskirts."""
    u = (jnp.arange(n) + 0.5) / n * 2.0 - 1.0
    return (half * jnp.sinh(a * u) / jnp.sinh(a),
            half * a * jnp.cosh(a * u) / jnp.sinh(a) * (2.0 / n))


_BX, _BW  = _sinh_nodes(BOX_HALF, 4.0, BOX_NXY)      # sky X (and Y)
_BS, _BSW = _sinh_nodes(BOX_SMAX, 7.0, BOX_NLOS)     # line of sight
_BY  = jnp.broadcast_to(_BX[:, None], (BOX_NXY, BOX_NLOS)).ravel()
_BSS = jnp.broadcast_to(_BS[None, :], (BOX_NXY, BOX_NLOS)).ravel()
_BWT = (_BW[:, None] * _BSW[None, :]).ravel()


def box_fractions(Rd_disc, hz_disc, L_bar, a_bar, b_bar, alpha, beta, gamma):
    """(p_disc, p_bar): the fraction of a UNIT-MASS disc, and of the unit-mass bar + bulge
    pair, that projects inside the sky box.  Angles in DEGREES, the convention of model.py:
    X = R @ x with R = makeRotationMatrix(alpha, beta, gamma), sky (X0, X1), line of sight X2.
    One sky column per lax.map step (~33k points), so it adds ~0.1 GB, not ~4 GB."""
    R    = makeRotationMatrix(alpha, beta, gamma)
    unit = {'logM': 0.0, 'Rd': Rd_disc, 'hz': hz_disc}

    def column(X):
        x, y, z = R.T @ jnp.stack([jnp.full_like(_BY, X), _BY, _BSS])
        rd = rho_expdisc(x, y, z, unit)
        rb = 0.5 * (T3_density(x, y, z, 1.0, a_bar, b_bar, L_bar, GAMMA_BAR)
                    + V4_density(x, y, z, 1.0, V4_A, V4_B, V4_L, V4_GAMMA))
        return jnp.stack([jnp.sum(rd * _BWT), jnp.sum(rb * _BWT)])

    p = jnp.sum(jax.lax.map(column, _BX) * _BW[:, None], axis=0)
    return p[0], p[1]


def stellar_masses(f_bar, log_mass_to_light_ratio, Rd_disc, hz_disc, L_bar, a_bar, b_bar,
                   alpha, beta, gamma):
    """-> (logM_disc, logM_bar, M_star, p_box).  M_star * p_box = M/L * L_BOX, then split:
    M_disc = (1 - f_bar) M_star, and M_bar = f_bar M_star / 2 because the T3 bar and the V4
    bulge EACH carry M_bar."""
    p_disc, p_bar = box_fractions(Rd_disc, hz_disc, L_bar, a_bar, b_bar, alpha, beta, gamma)
    p_box  = (1.0 - f_bar) * p_disc + f_bar * p_bar
    M_star = 10.0 ** log_mass_to_light_ratio * L_BOX / p_box
    return (jnp.log10((1.0 - f_bar) * M_star), jnp.log10(0.5 * f_bar * M_star), M_star, p_box)


# ================================================================= the 12 parameters
# The same 12 as the Miyamoto-Nagai notebook; the disc scale length and scale height take the
# slots of Rs and Hs.
param_names = ['logM_10kpc', 'f_bar', 'logC_halo', 'logRd_disk',
               'loghz_disk', 'logL_bar', 'alpha', 'beta', 'gamma',
               'log_mass_to_light_ratio', 'log_Omega', 'log10_lambda']
NDIM  = len(param_names)
J_LAM = param_names.index('log10_lambda')      # NEVER hard-code this index


@partial(jax.jit, static_argnames=('num_Vbin', 'Rzphi_n_tot', 'n_boot'))
def run_model(theta, dict_data, num_Vbin, Rzphi_n_tot, n_boot=N_BOOT_MEFF):
    """The forward model: the parent's bar, bulge and halo, NGC 1512's double-exponential disc
    with a CylSpline potential, and the stellar mass derived from M/L."""
    logM_enc, log_c = theta[0], theta[2]
    logM_halo, logRs_halo = logMenc_logc_to_logM_logRs(logM_enc, log_c, r_enc=10.0,
                                                       Delta=200., rho_crit=277.54)
    f_bar = theta[1]
    logRd_disk, loghz_disk, logL_bar = theta[3], theta[4], theta[5]
    alpha = theta[6] * 180 / jnp.pi
    beta  = theta[7] * 180 / jnp.pi
    gamma = theta[8] * 180 / jnp.pi
    log_mass_to_light_ratio, log_Omega_bar = theta[9], theta[10]

    L_bar   = 10.0 ** logL_bar
    a_bar   = L_bar / 5.0
    Rd_disc = 10.0 ** logRd_disk
    hz_disc = 10.0 ** loghz_disk
    b_bar   = hz_disc                   # the bar is as thick as the disc, as in the parent

    logM_disc, logM_bar, M_star, p_box = stellar_masses(
        f_bar, log_mass_to_light_ratio, Rd_disc, hz_disc, L_bar, a_bar, b_bar,
        alpha, beta, gamma)

    params_halo_pot = {'logM': logM_halo, 'Rs': 10 ** logRs_halo, 'a': 1.0, 'b': 1.0, 'c': 1.0,
                       'x_origin': 0.0, 'y_origin': 0.0, 'z_origin': 0.0,
                       'dirx': 0.0, 'diry': 0.0, 'dirz': 1.0}
    params_baryon_rho = {'logM_disc': logM_disc, 'Rd_disc': Rd_disc, 'hz_disc': hz_disc,
                         'logM_bar': logM_bar, 'L_bar': L_bar,
                         'a_bar': a_bar, 'b_bar': b_bar,
                         'mass_to_light_ratio': 10 ** log_mass_to_light_ratio,
                         'Omega_bar': 10 ** log_Omega_bar,
                         'x_origin': 0.0, 'y_origin': 0.0, 'z_origin': 0.0,
                         'dirx': 0.0, 'diry': 0.0, 'dirz': 1.0,
                         'alpha': alpha, 'beta': beta, 'gamma': gamma}

    # the five callables, closing over THIS theta's disc spline and bar table (cell 4)
    dens, pot, acc, pot_axi, dens_axi = make_model_funcs(params_baryon_rho, params_halo_pot)

    X_minmax, Y_minmax = dict_data['X_minmax'], dict_data['Y_minmax']
    nX, nY = dict_data['nX_nY']

    # anisotropy_b is passed as a KEYWORD.  Patching jeans.get_w0_new_improved does NOT work:
    # build_model_acc_fourier forwards `anisotropy_b=anisotropy_b` explicitly with a default of
    # 1.0, and an explicit keyword overrides a functools.partial binding.
    # lambda stays TRACED (it arrives inside theta); as a static argname it would recompile on
    # EVERY MCMC step.
    out = build_model_acc_meff_fourier(
        dens, pot, acc, pot_axi, dens_axi,
        params_halo_pot, params_baryon_rho, dict_data, num_Vbin,
        anisotropy_b=ANISO_B, n_realizations=N_REALIZATIONS,
        sigma_frac_3d=SIG_FRAC_An, sigma_frac_m0=SIG_FRAC_A0, sigma_scheme=F3D_SCHEME,
        floor_frac_3d=F3D_FLOOR, v_sys=0.0,
        mmax=F3D_MMAX, nR_f=F3D_R[2], nZ_f=F3D_Z[2], log_R=F3D_LOG, log_z=F3D_LOG,
        R_lo_f=F3D_R[0], R_hi_f=F3D_R[1], z_lo_f=F3D_Z[0], z_hi_f=F3D_Z[1],
        n_phi_t=F3D_NPHI_T, n_sub_t=F3D_NSUB_T,
        pow_R=F3D_POW_R, pow_z=F3D_POW_Z, close_inner=F3D_CLOSE_INNER,
        Rzphi_n_tot=Rzphi_n_tot, Rzphi_n_grid=dict_data['Rzphi_n_grid'],
        Rzphi_lim_grid=jnp.array([dict_data['R_minmax'], dict_data['z_minmax'],
                                  dict_data['phi_minmax']]),
        xy_lim_grid=jnp.array([X_minmax, Y_minmax]), xy_n_grid=jnp.array([nX, nY]),
        nnls_maxiter=200, regularization=10.0 ** theta[J_LAM],
        err_den_meff=ERR_DEN_MEFF, n_boot_meff=n_boot)
    return dict(out, M_star=M_star, p_box=p_box, logM_disc=logM_disc, logM_bar=logM_bar)


@partial(jax.jit, static_argnames=('num_Vbin', 'Rzphi_n_tot'))
def model_likelihood(theta, dict_data, num_Vbin, Rzphi_n_tot, sigma_amplify=1.0):
    """logL = -0.5 * (chi2_kin + 2*m_eff_kin + chi2_den) / sigma_amplify**2.

    NO log-mean-exp: this is a SINGLE unperturbed fit.  The bootstraps are spent on m_eff.
    The SURFACE DENSITY is in chi2 but is NOT charged m_eff: measured on NGC 1512 it carries
    23% of the mean chi2 but 88% of its variance, and removing it from the AIC puts sigma_AIC
    on its Gaussian floor.  There is no gas term -- the mock has no rotation curve.
    """
    res = run_model(theta, dict_data, num_Vbin, Rzphi_n_tot)

    O_den = dict_data['XY_density_data']
    sig_l = (ERR_DEN_LOGL * O_den) / (O_den * jnp.log(10) + EPSILON)
    mden  = res['surface_density'][0]
    chi2_den = jnp.nansum(((jnp.log10(mden + EPSILON) - jnp.log10(O_den + EPSILON)) / sig_l) ** 2)

    chi2_kin = 0.0
    for k in (1, 2, 3, 4):
        chi2_kin = chi2_kin + jnp.nansum(
            ((res['h%d' % k][0] - dict_data['h%d_data' % k])
             / (dict_data['h%d_data_err' % k] + EPSILON)) ** 2)

    aic = chi2_kin + 2.0 * res['m_eff_kin']
    return -0.5 * (aic + chi2_den) / sigma_amplify ** 2


def chi2_report(theta):
    """Per-bin chi2 and the AIC terms, as plain numbers."""
    res = run_model(jnp.asarray(theta), dict_data, NUM_VBIN, RZ_TOT)
    O = np.asarray(dict_data['XY_density_data'])
    d0 = np.asarray(res['surface_density'])[0]
    sl = (ERR_DEN_LOGL * O) / (O * np.log(10) + 1e-30)
    out = {'logSig': float(np.nanmean(((np.log10(d0 + 1e-30) - np.log10(O + 1e-30)) / sl) ** 2))}
    kin = 0.0
    for k in (1, 2, 3, 4):
        r = ((np.asarray(res['h%d' % k])[0] - np.asarray(dict_data['h%d_data' % k]))
             / np.asarray(dict_data['h%d_data_err' % k])) ** 2
        out['h%d' % k] = float(np.nanmean(r)); kin += float(np.nansum(r))
    out['SUM']        = sum(out[k] for k in ('logSig', 'h1', 'h2', 'h3', 'h4'))
    out['m_eff_kin']  = float(res['m_eff_kin'])
    out['AIC']        = kin + 2 * out['m_eff_kin']
    out['flux_d/m']   = float(O.sum() / d0.sum())
    out['logM_star']  = float(np.log10(res['M_star']))     # derived, not sampled
    out['logM_disc']  = float(res['logM_disc'])
    out['logM_bar']   = float(res['logM_bar'])
    out['p_box']      = float(res['p_box'])
    return out


# Example code for calling the likelihood function

The mock's generating parameters are known, so these scans have a right answer. The halo, the
viewing angles, $\Omega$ and $M/L$ are the mock's own, exactly as in the Miyamoto–Nagai notebook.
The **disc and bar shapes and $f_{\rm bar}$** are this family's: the parent's cell 7 is a
Miyamoto–Nagai fit and does not describe an exponential disc. They come from a direct
least-squares fit of **this** family — double-exponential disc + T3 bar + V4 bulge, with
`run_model`'s ties — to the N-body star particles: the stellar $v_c(R)$ at 1%, the face-on
$\Sigma(R)$ at 0.05 dex and $A_2/A_0(R)$ at 0.02 (`fit_family_to_nbody.py`, family `exp_tied`).

**One exception: $L_{\rm bar}$ is the published example's 4.87 kpc** (its cell 7, which the
Miyamoto–Nagai notebook also uses), not the 5.34 kpc of this fit, so all three notebooks share one
bar length. $L_{\rm bar}$ is a parameter of the T3 model, not a measured bar length. With this fit's
other shapes, 4.87 kpc puts the $A_2/A_0$ peak (0.37) at 2.6 kpc, and it falls to half by 4.8 kpc.
The N-body peaks at 0.40 at 3.3 kpc and falls to half by 5.4 kpc.

$\log M/L=0$ is exact: the mock's surface density *is* the N-body's stellar mass. The shapes are a
parametric description, and an imperfect one: no single smooth disc follows the N-body's disc,
whose local scale length grows from 2.4 kpc (2–6 kpc) to 9.4 kpc (4–9 kpc). At these shapes the
model needs 28% more mass in the box than the light ($\log M/L=+0.108$) to match the N-body's
stellar potential. `chi2_report` prints the derived $M_\star$ and $p_{\rm box}$.

In [ ]:
# The disc and bar shapes and f_bar: a direct fit of THIS family (exponential disc + T3 bar +
# V4 bulge, run_model's ties) to the N-body star particles -- v_c,*(R) 1%, face-on Sigma 0.05 dex,
# A2/A0 0.02 (fit_family_to_nbody.py, family exp_tied).  Halo, angles, Omega, M/L: the mock's own.
# EXCEPT L_bar: the published example's 4.87 kpc (example_schwarmax.ipynb cell 7), shared by all three
# notebooks; this fit gave 5.343.  L_bar barely moves the derived masses (p_box, p_field change < 1e-3).
_Md, _Mb = 5.37301e10, 1.60154e10    # disc; bar (= bulge)  ->  f_bar = 2 M_bar / M_star
ground_truth = [
  10.76,                       # logM_10kpc
  2 * _Mb / (_Md + 2 * _Mb),   # f_bar = 0.3735
  np.log10(7.76),              # logC_halo
  np.log10(6.466),             # logRd_disk  [kpc]
  np.log10(0.7477),            # loghz_disk  [kpc]
  np.log10(4.87),              # logL_bar    [kpc]  (the published example's; this fit gave 5.343)
  np.radians(25.0),            # alpha
  np.radians(65.0),            # beta   (inclination)
  np.radians(45.0),            # gamma
  0.0,                         # log_mass_to_light_ratio
  1.40,                        # log_Omega
]
theta_true = jnp.asarray(ground_truth + [0.0])      # log10(lambda) = 0 -> lambda = 1

t0 = time.time()
rep = chi2_report(theta_true)
print('chi2_report at GROUND TRUTH  (%.0f s)' % (time.time() - t0))
for k, v in rep.items():
    print('   %-12s %10.4f' % (k, v))
print('\nlogL at truth (S2 = %.2f): %.4f'
      % (S2, float(model_likelihood(theta_true, dict_data, NUM_VBIN, RZ_TOT, jnp.sqrt(S2)))))


In [ ]:
# The old notebook scans M/L and Omega; same two here, plus the inclination, which is the
# parameter model-flexibility bias corrupts most and the reason for using AIC at all.
fig, ax = plt.subplots(1, 3, figsize=(16, 3.4), constrained_layout=True)
for a, (j, lo, hi, lab) in zip(ax, [(param_names.index('log_mass_to_light_ratio'), -0.6, 0.3, 'log M/L'),
                                    (param_names.index('log_Omega'), 1.0, 1.8, 'log Omega'),
                                    (param_names.index('beta'), 0.9, 1.4, 'beta [rad]')]):
    xs = np.linspace(lo, hi, 9)
    ll = []
    for x in tqdm(xs, desc=lab, leave=False):
        th = np.asarray(theta_true).copy(); th[j] = x
        ll.append(float(model_likelihood(jnp.asarray(th), dict_data, NUM_VBIN, RZ_TOT,
                                         jnp.sqrt(S2))))
    ll = np.asarray(ll)
    a.plot(xs, ll - np.nanmax(ll), 'o-')
    a.axvline(float(np.asarray(theta_true)[j]), color='crimson', ls='--', label='truth')
    a.set_xlabel(lab); a.set_ylabel(r'$\log L - \max$'); a.legend(fontsize=8)
    a.set_title('peak at %.3f (truth %.3f)' % (xs[int(np.nanargmax(ll))],
                                               float(np.asarray(theta_true)[j])), fontsize=9)
plt.show()


# Optimisation pipeline

`N_PART` particles are hill-climbed together with a cooling step, exactly as in the tested
NGC 1512 chains. Every iteration evaluates all of them in **one vmapped call**, so the progress
bar below counts **iterations** (`N_ITER` of them), not starting points, and each tick is
`N_PART` likelihood evaluations.

The first batched call compiles the whole likelihood and is timed and printed *before* the bar
appears, so the initial wait is accounted for rather than looking like a hang. After that a tick
should arrive every `N_PART` x (seconds per evaluation) — if the bar sits at 0/`N_ITER` for much
longer than the compile line reported, something really is wrong.

In [ ]:
# Multi-start optimisation.  The whole ENSEMBLE is saved, not just the best point: the dAIC
# calibration below needs its spread to place the second model.
#
# This is `parallel_minimize` from the tested NGC 1512 chains: a cooled random-walk hill climb
# in which every iteration evaluates all N_PART particles in ONE vmapped call.  It replaces a
# serial scipy Nelder-Mead for two measured reasons.
#
#   PROGRESS.  Nelder-Mead is serial, and `maxiter = 40*NDIM` rails at 520 iterations for
#   672-705 function calls per start (measured, scipy 1.17; `maxfev` is left at infinity when
#   only `maxiter` is set, so nothing else bounds it).  A bar over 8 starts therefore advances
#   once per ~690 likelihood calls.  At the 2.0 s/eval this model costs on a T4 -- 63.84 s/step
#   for 32 chains, from the parent notebook's own saved output -- that is 23 min per tick with
#   nothing to watch, and 5498 evaluations for the cell.  Here the bar ticks once per iteration,
#   N_PART evaluations apart, and the cell costs N_PART*(N_ITER+1) = 968.
#
#   CONSISTENCY.  vmapped and serial evaluation are NOT the same number in float32.  Both are
#   bit-reproducible and batch-content-independent, but the NNLS lands on a different point of
#   its degenerate ridge: at the truth, 5 of 159 active orbits differ, sum(w) by 0.29%, and
#   chi2_kin +8.9% against chi2_den -10.0% (they partly cancel, to ~15 in logL).  emcee below
#   evaluates its walkers in vmapped chunks, so a serially-optimised start is the optimum of a surface the
#   sampler never evaluates.  Optimise on the surface you sample.
RUN_MINIMISER = True
N_PART     = 8      # particles, all evaluated in one batched call
N_ITER     = 120    # iterations; the tested NGC 1512 value.  Step cools as 0.97**it -> x0.026
VMAP_BATCH = 8      # <<< TUNE ME: evaluations per batched call -- the same budget as
                    # EMCEE_CHUNK below.  The Miyamoto-Nagai notebook ran 8 at once on a 24 GB
                    # L4; the disc spline adds to that (not yet measured on a GPU).  Lower it
                    # on a smaller card or on an out-of-memory error.  The bar still ticks once per ITERATION (N_PART
                    # evaluations) whatever this is set to.
MINIMISER_RESULT = os.path.join(path, 'example/minimisation_result_%s.npy' % RUN_TAG)
MINIMISER_POS    = os.path.join(path, 'example/minimiser_pos_%s.npy' % RUN_TAG)
MINIMISER_VAL    = os.path.join(path, 'example/minimiser_val_%s.npy' % RUN_TAG)

# Per-parameter step: the Miyamoto-Nagai notebook's tested BlackJAX `rmh_sigma_init`.  f_bar is a fraction: 0.02 absolute.
OPT_SPREAD = np.array([0.05, 0.02, 0.02, 0.02, 0.02, 0.02,
                       0.02, 0.02, 0.07, 0.02, 0.02, 0.20])
# Search box: the shape of the MCMC prior, centred on the generating parameters rather than on
# the minimiser result (which does not exist yet).  f_bar stays off 0 and 1, where log M_bar or
# log M_disc would be -inf.
_t = np.asarray(theta_true, float)
OPT_LO = np.array([_t[0] - 3, 0.01, _t[2] - 1, _t[3] - 1, _t[4] - 1,
                   max(0.0, _t[5] - 1), 0., 0., 0., -2., 0., -1.])
OPT_HI = np.array([_t[0] + 3, 0.99, _t[2] + 1, _t[3] + 1, _t[4] + 1,
                   min(1.0, _t[5] + 1), np.pi, np.pi / 2, np.pi, 2., 2., 4.])
assert OPT_SPREAD.size == NDIM and OPT_LO.size == NDIM and OPT_HI.size == NDIM

_lp      = lambda th: model_likelihood(th, dict_data, NUM_VBIN, RZ_TOT, jnp.sqrt(S2))
_lp_vmap = jax.vmap(_lp)

def parallel_minimize(x0s, spread, n_iter, lo, hi, vmap_batch=VMAP_BATCH, seed=0):
    """Cooled random-walk hill climb, batched over particles (tested NGC 1512 structure)."""
    lo, hi = jnp.asarray(lo), jnp.asarray(hi)
    def ev(p):
        return jnp.concatenate([_lp_vmap(p[i:i + vmap_batch])
                                for i in range(0, p.shape[0], vmap_batch)])
    key = jax.random.PRNGKey(seed)
    pos = jnp.clip(jnp.asarray(x0s), lo, hi)
    t0  = time.time(); val = ev(pos); val.block_until_ready()
    print('   first batched call (JIT compile + %d evaluations): %.1f s  -- the bar below ticks'
          % (pos.shape[0], time.time() - t0))
    print('   once per iteration, %d evaluations apart; %d iterations in total.'
          % (pos.shape[0], n_iter))
    sp  = jnp.asarray(spread)
    bar = tqdm(range(n_iter), desc='minimiser', unit='iter')
    for it in bar:
        key, k1 = jax.random.split(key)
        cand   = jnp.clip(pos + jax.random.normal(k1, pos.shape) * sp * (0.97 ** it), lo, hi)
        cv     = ev(cand)
        better = cv > val
        pos    = jnp.where(better[:, None], cand, pos)
        val    = jnp.where(better, cv, val)
        # float() forces a device sync, so the bar and its ETA are honest
        bar.set_postfix(best='%.2f' % float(jnp.max(val)),
                        moved='%d/%d' % (int(better.sum()), pos.shape[0]))
    return np.asarray(pos), np.asarray(val)

if RUN_MINIMISER:
    rng   = np.random.default_rng(0)
    start = _t[None, :] + OPT_SPREAD[None, :] * rng.standard_normal((N_PART, NDIM))
    t0    = time.time()
    pos, val = parallel_minimize(start, OPT_SPREAD, N_ITER, OPT_LO, OPT_HI)
    print('   %d iterations x %d particles = %d evaluations in %.0f s (%.2f s/eval)'
          % (N_ITER, N_PART, N_PART * (N_ITER + 1), time.time() - t0,
             (time.time() - t0) / (N_PART * (N_ITER + 1))))
    np.save(MINIMISER_POS, pos); np.save(MINIMISER_VAL, val)
    np.save(MINIMISER_RESULT, pos[int(np.argmax(val))])
else:
    pos = np.load(MINIMISER_POS); val = np.load(MINIMISER_VAL)

best = pos[int(np.argmax(val))]
print('best logL %.4f  (ensemble spread %.4f over %d particles)'
      % (val.max(), float(np.std(val)), len(val)))
print('%-26s %11s %11s %11s' % ('parameter', 'truth', 'best fit', 'offset'))
for j, n in enumerate(param_names):
    t = float(np.asarray(theta_true)[j])
    print('%-26s %11.4f %11.4f %+11.4f' % (n, t, best[j], best[j] - t))
_jb = param_names.index('beta')
print('\n   i_true = %.2f deg | i_fit = %.2f deg'
      % (np.degrees(float(np.asarray(theta_true)[_jb])), np.degrees(best[_jb])))


# The $\chi^2$ scale

The old notebook set `sigma_amplify` from a **jackknife** $\Delta\chi^2$. This cell replaces it
with the two quantities Wang, Thomas & Lipka (2026) measure. They answer different questions and
only one of them is a temperature.

**1. Is the noise model right?** $\sigma_{\rm AIC}$, the scatter of a *single* model's AIC over
noise realisations, should sit near the Gaussian floor $\sqrt{2\chi^2}$ — Wang+2026 measure a
ratio of 1.13. If it does, the familiar $S^2=\sqrt{2N_{\rm kin}}$ is justified rather than
assumed.

**2. How big a $\Delta$AIC is significant?** Not $\sigma_{\rm AIC}$: two models evaluated on the
*same* noise realisations share most of it. The paired scatter is
$$\sigma_{\Delta\rm AIC}=\sigma_{\rm AIC}\sqrt{2(1-\rho)}\,,$$
which **vanishes as the two models approach each other** ($\rho\to1$). So there is no single
$\sigma_{\Delta\rm AIC}$: it is a property of a model *pair*, and Wang+2026 accordingly quote
one number for one pair of competing mass models, as a model-**selection** threshold.

The cell walks one fixed direction in parameter space and tabulates $\rho$, $|\Delta{\rm AIC}|$
and $\sigma_{\Delta\rm AIC}$, so the pair-dependence is visible rather than hidden.

> **Why $\sigma_{\Delta\rm AIC}$ is NOT adopted as the temperature here.** It is tempting to
> look for the displacement where $|\Delta{\rm AIC}|=\sigma_{\Delta\rm AIC}$ and call that the
> $1\sigma$ contour. **Measured on this mock (1000 orbits, 16 realisations, displacements
> 0.125–4 spreads), it does not work:** $\sigma_{\Delta\rm AIC}$ rises smoothly, 11 → 28, but
> $|\Delta{\rm AIC}|$ does not grow at all — 25.5, 26.4, 41.7, 23.0, 10.8, 75.0, a fitted slope
> of $s^{0.09}$ where a quadratic would give $s^{2}$. At this orbit count the AIC landscape is
> rough on the scale being probed, so any crossing found by interpolation is noise. **Check
> your own table before trusting a measured $S^2$**: $|\Delta{\rm AIC}|$ must grow
> monotonically first. `ADOPT` defaults to the Gaussian value for this reason.

In [ ]:
N_CAL  = 24                            # noise realisations per model
SCALES = (0.25, 0.5, 1.0, 2.0, 4.0)    # displacements, in units of the minimiser spread
ADOPT  = 'gaussian'                    # 'gaussian' = sqrt(2 N_kin); 'measured' = sigma_dAIC,
                                       # which the markdown above says to verify FIRST

# ONE fixed random direction, then walked outwards.  A fresh direction per scale would confound
# the displacement with the direction.
spread = np.std(pos, axis=0) + 1e-6
unit   = np.random.default_rng(20260922).standard_normal(NDIM) * spread
unit[J_LAM] = 0.0      # lambda is IDENTICAL in every pair: it is a property of the estimator,
                       # not of the model, so comparing two lambdas is a different experiment.
thA = best.copy()

# The noise realisations are the ones ALREADY in dict_data (rows 1..N_CAL; row 0 is zeroed by
# load_data_bootstrap and is skipped by the m_eff path).  Reusing the stored rows rather than
# drawing fresh ones is what makes the models PAIRED -- the common-mode noise then cancels in
# aA - aB, which is the entire reason sigma_dAIC is smaller than sigma_AIC.
_NROW = int(np.asarray(dict_data['XY_standard_normal']).shape[0]) - 1
assert N_CAL <= _NROW, 'only %d stored noise rows; raise N_BOOTSTRAP in load_data_bootstrap' % _NROW


def _realisations(th):
    """(chi2 per realisation, m_eff) -- kept apart so the Gaussian floor uses chi2 alone."""
    r = run_model(jnp.asarray(th), dict_data, NUM_VBIN, RZ_TOT, n_boot=N_CAL)
    return np.asarray(r['chi2_boot_kin'], float), float(r['m_eff_kin'])


t0 = time.time()
c2A, meA = _realisations(thA)
aA      = c2A + 2.0 * meA
SIG_AIC = float(aA.std(ddof=1))
FLOOR   = float(np.sqrt(2.0 * c2A.mean()))      # on chi2, NOT on chi2 + 2 m_eff
print('QUESTION 1 -- is the noise model right?   (%d realisations)' % N_CAL)
print('   <chi2_kin> %.1f over %d points | m_eff_kin %.2f' % (c2A.mean(), N_KIN, meA))
print('   sigma_AIC (one model)        %9.2f' % SIG_AIC)
print('   Gaussian floor sqrt(2 chi2)  %9.2f' % FLOOR)
print('   RATIO                        %9.2f    [Wang+2026: 1.13]' % (SIG_AIC / FLOOR))
print('   sqrt(2 N_kin)                %9.2f    <- the provisional S2' % S2)

print('\nQUESTION 2 -- how big a dAIC is significant?')
print('   %6s %10s %10s %11s %8s %10s' % ('scale', 'rho', '|dAIC|', 'sig_dAIC', 'ratio', 'predicted'))
rows = []
for sc in SCALES:
    thB = thA + sc * unit
    thB[J_LAM] = thA[J_LAM]
    c2B, meB = _realisations(thB)
    aB  = c2B + 2.0 * meB
    d   = abs(float(aA.mean() - aB.mean()))
    sd  = float((aA - aB).std(ddof=1))
    rho = float(np.corrcoef(aA, aB)[0, 1])
    rows.append((sc, rho, d, sd, d / max(sd, 1e-12)))
    print('   %6.2f %10.6f %10.2f %11.2f %8.3f %10.2f'
          % (rows[-1] + (SIG_AIC * np.sqrt(max(2 * (1 - rho), 0.0)),)))
rows = np.asarray(rows)
print('   the last column is sigma_AIC*sqrt(2(1-rho)) -- it should track column 4')
print('   (%.0f s for %d model evaluations)' % (time.time() - t0, 1 + len(SCALES)))

# ---- is |dAIC| even growing?  the markdown explains why this is the gate ---------------
_pl = lambda c: float(np.polyfit(np.log(rows[:, 0]), np.log(np.maximum(rows[:, c], 1e-12)), 1)[0])
p_d, p_s = _pl(2), _pl(3)
mono = bool(np.all(np.diff(rows[:, 2]) > 0))
print('\n   |dAIC|   ~ s^%+.2f   (a quadratic landscape gives +2)' % p_d)
print('   sig_dAIC ~ s^%+.2f' % p_s)
print('   |dAIC| monotonically increasing: %s' % mono)

S2_MEASURED = np.nan
if rows[0, 1] > 0.9999:
    print('   -> rho = %.6f even at the smallest displacement: the two models are NOT distinct,'
          % rows[0, 1])
    print('      which is an unconverged minimiser rather than a real resolution.')
elif not mono or p_d < 1.0:
    print('   -> |dAIC| is NOT growing with displacement, so no crossing is meaningful.')
    print('      This is what we measured on this mock; see the markdown above.')
elif rows[:, 4].min() < 1.0 < rows[:, 4].max():
    _o = np.argsort(rows[:, 4])
    s_star = float(np.exp(np.interp(0.0, np.log(rows[_o, 4]), np.log(rows[_o, 0]))))
    S2_MEASURED = float(np.exp(np.interp(np.log(s_star), np.log(rows[:, 0]), np.log(rows[:, 3]))))
    print('   -> |dAIC| = sigma_dAIC at %.3f spreads  ->  sigma_dAIC = %.2f' % (s_star, S2_MEASURED))
    for j, n in enumerate(param_names):
        if abs(unit[j]) > 0:
            print('      1-sigma %-26s %+.4f' % (n, s_star * unit[j]))
else:
    print('   -> |dAIC|/sigma_dAIC never crosses 1 over %s; widen SCALES.' % (list(SCALES),))

print('\nsqrt(2 N_kin) = %.2f   |   measured sigma_dAIC = %s'
      % (S2, 'n/a' if not np.isfinite(S2_MEASURED) else '%.2f' % S2_MEASURED))
if ADOPT == 'measured' and np.isfinite(S2_MEASURED):
    S2_ADOPTED = S2_MEASURED; print('ADOPTED: the measured sigma_dAIC (ADOPT = %r)' % ADOPT)
else:
    S2_ADOPTED = S2
    print('ADOPTED: sqrt(2 N_kin) (ADOPT = %r%s)'
          % (ADOPT, '' if ADOPT == 'gaussian' else '; no usable measured value'))
SIGMA_AMPLIFY = jnp.sqrt(S2_ADOPTED)
print('S2 in force for the chain: %.2f   (posterior widths scale as sqrt(S2))' % S2_ADOPTED)


# MCMC

**emcee**, the affine-invariant ensemble sampler, with differential-evolution (80%) and snooker
(20%) moves — the cell of the NGC 1512 notebooks. It replaces BlackJAX's adaptive random-walk
Metropolis, which stalled on NGC 1512 (acceptance 0.41 → 0.04, $\hat R$ 1.3–2.3, a proposal 3–9×
wider than the posterior) and ran at acceptance 0.125 with $\hat R$ 1.19–1.49 after 690 steps on
this mock. Affine-invariant moves need no tuned covariance, which is the part that failed.

`vectorize=True` hands each half-ensemble to `log_prob_vec` in one call, evaluated in vmapped
chunks of `EMCEE_CHUNK` (the memory knob). The chain is written to an HDF5 file after every step:
with `RESUME = True` a re-run continues it up to `EMCEE_STEPS` in total, so a Colab disconnect
loses nothing.

In [ ]:
# ===========================================================================
# MCMC: emcee affine-invariant ensemble (NGC 1512's cell).
# vectorize=True gives log_prob_vec the whole half-ensemble in ONE call, so jax.vmap works; the
# call is CHUNKED so peak memory is set by EMCEE_CHUNK, not by the walker count.
# ===========================================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'emcee'], check=False)
import emcee
print('emcee', emcee.__version__)

EMCEE_WALKERS = 32        # >= 2*NDIM = 24 (emcee refuses fewer); a multiple of 4*EMCEE_CHUNK
EMCEE_STEPS   = 700      # TOTAL chain length, not steps-per-run: with RESUME the cell tops the
                          # stored chain up to this number
EMCEE_CHUNK   = 8         # <<< TUNE ME: evaluations held in flight at once (memory knob).  Keep
                          # it a divisor of EMCEE_WALKERS // 4: the snooker move updates the
                          # walkers in 4 groups, and a short group is padded to a full chunk.
                          # Lower it on an out-of-memory error.
EMCEE_FILE    = os.path.join(path, 'example/emcee_%s.h5' % RUN_TAG)
OUTPUT_FILE   = os.path.join(path, 'example/emcee_results_%s.pkl' % RUN_TAG)
RESUME        = True      # True  -> continue the chain in EMCEE_FILE (an empty or missing file
                          #          starts at step 0).  The backend flushes every step.
                          # False -> start over, DELETING that file's contents.

res = np.load(MINIMISER_RESULT)

# ---- Prior bounds (12D): identical to the Miyamoto-Nagai notebook -------------------------
# f_bar: uniform on [0.01, 0.99], kept off 0 and 1 where log M_bar or log M_disc is -inf.
BOUNDS_LO = jnp.array([res[0] - 3, 0.01, res[2] - 1, res[3] - 1,
                       res[4] - 1, max(0, res[5] - 1),
                       0., 0., 0., -2., 0., -1.])
BOUNDS_HI = jnp.array([res[0] + 3, 0.99, res[2] + 1, res[3] + 1,
                       res[4] + 1, min(1, res[5] + 1),
                       float(jnp.pi), float(jnp.pi / 2), float(jnp.pi), 2., 2., 4.])
assert BOUNDS_LO.size == NDIM and BOUNDS_HI.size == NDIM


def logdensity_fn(theta):
    in_bounds = jnp.all((theta >= BOUNDS_LO) & (theta <= BOUNDS_HI))
    log_vol   = jnp.sum(jnp.log(BOUNDS_HI - BOUNDS_LO))
    logprior  = jnp.where(in_bounds, -log_vol, -jnp.inf)
    ll = model_likelihood(theta, dict_data, NUM_VBIN, RZ_TOT, sigma_amplify=SIGMA_AMPLIFY)
    ll = jnp.where(jnp.isfinite(ll), ll, -1e30)
    return logprior + ll


_lik = jax.jit(jax.vmap(logdensity_fn))


def log_prob_vec(P):
    """P is (n, NDIM) from emcee (vectorize=True) -> (n,) log-probabilities, in chunks of
    EMCEE_CHUNK.  A short last chunk is PADDED to EMCEE_CHUNK, so there is one compiled shape
    and no recompilation.  A model that failed (-1e30) is returned as -inf."""
    P = np.atleast_2d(np.asarray(P, dtype=np.float32))
    out = np.empty(P.shape[0], dtype=np.float64)
    for a in range(0, P.shape[0], EMCEE_CHUNK):
        blk = P[a:a + EMCEE_CHUNK]
        n = blk.shape[0]
        if n < EMCEE_CHUNK:
            blk = np.concatenate([blk, np.repeat(blk[-1:], EMCEE_CHUNK - n, axis=0)])
        out[a:a + n] = np.asarray(_lik(jnp.asarray(blk)), dtype=np.float64)[:n]
    return np.where(np.isfinite(out) & (out > -1e29), out, -np.inf)


print('initial parameters (the minimiser result):')
for i in range(NDIM):
    print('   %-26s %.6f' % (param_names[i], res[i]))

_lo, _hi = np.asarray(BOUNDS_LO, float), np.asarray(BOUNDS_HI, float)
_moves   = [(emcee.moves.DEMove(), 0.8), (emcee.moves.DESnookerMove(), 0.2)]
_backend = emcee.backends.HDFBackend(EMCEE_FILE)


def _stored_steps(b):
    """Steps already in the backend.  emcee's .iteration RAISES on a file that does not exist
    yet (the first run) or that reset() never initialised, rather than returning 0."""
    try:
        return b.iteration
    except (OSError, KeyError):
        return 0


if RESUME and _stored_steps(_backend) > 0:
    # get_last_sample() carries the coordinates, the log-probabilities AND the RNG state, so this
    # is a true continuation.  The walker count comes from the FILE: emcee refuses a sampler
    # whose shape disagrees with its backend.
    _done = _backend.iteration
    EMCEE_WALKERS = _backend.shape[0]
    _p0 = _backend.get_last_sample()
    print('RESUME: %d steps already stored (%d walkers), topping up to EMCEE_STEPS = %d'
          % (_done, EMCEE_WALKERS, EMCEE_STEPS))
else:
    _done = 0
    # Start in a ball around the minimiser result, REFLECTED off the prior box rather than
    # clipped (clipping stacks walkers on a bound, where the prior returns -inf).  The ball is
    # the Miyamoto-Nagai notebook's BlackJAX initial_guess_sigma.
    EMCEE_BALL = np.array([0.05, 0.03, 0.02, 0.02, 0.02, 0.02,
                           0.02, 0.02, 0.10, 0.05, 0.10, 0.30])
    assert EMCEE_BALL.size == NDIM
    _start = np.asarray(res, dtype=np.float64)
    _eps = 1e-4 * (_hi - _lo)         # survives the float32 cast in log_prob_vec
    _oob = np.where((_start <= _lo) | (_start >= _hi))[0]
    assert _oob.size == 0, ('the start point is outside the prior in ' + ', '.join(
        '%s=%.4f not in [%.4f, %.4f]' % (param_names[k], _start[k], _lo[k], _hi[k]) for k in _oob))

    def _reflect(x, lo, hi):
        """Fold x back into [lo, hi] by repeated mirror reflection (a triangle wave)."""
        w = hi - lo
        y = np.mod(x - lo, 2.0 * w)
        return lo + np.where(y > w, 2.0 * w - y, y)

    _rng = np.random.default_rng(0)
    _p0  = np.clip(_reflect(_start[None, :] + EMCEE_BALL[None, :]
                            * _rng.standard_normal((EMCEE_WALKERS, NDIM)), _lo, _hi),
                   _lo + _eps, _hi - _eps)
    _chk = log_prob_vec(_p0)
    # a walker whose model will not solve is re-drawn from a ball that halves each round
    for _try in range(1, 21):
        _bad = ~np.isfinite(_chk)
        if not _bad.any():
            break
        _cand = np.clip(_reflect(_start[None, :] + (0.5 ** _try * EMCEE_BALL)[None, :]
                                 * _rng.standard_normal((int(_bad.sum()), NDIM)), _lo, _hi),
                        _lo + _eps, _hi - _eps)
        _p0[_bad] = _cand
        _chk[_bad] = log_prob_vec(_cand)
        print('   re-drew %d walker(s) at %.1f%% of the ball' % (_bad.sum(), 100 * 0.5 ** _try))
    assert np.isfinite(_chk).all(), 'walkers still failing after 20 re-draws -- inspect _p0[~np.isfinite(_chk)]'
    print('initial ensemble: %d walkers, logP %.2f .. %.2f' % (EMCEE_WALKERS, _chk.min(), _chk.max()))
    _backend.reset(EMCEE_WALKERS, NDIM)          # DELETES whatever was in the file

_sampler = emcee.EnsembleSampler(EMCEE_WALKERS, NDIM, log_prob_vec,
                                 vectorize=True, moves=_moves, backend=_backend)

# The bar is driven here rather than by progress=True so that it CONTINUES from the stored step
# count.  Status lines go through tqdm.write: a bare print inside the loop tears the bar apart.
t0 = time.time()
_todo = max(0, EMCEE_STEPS - _done)
if _todo == 0:
    print('chain already has %d steps >= EMCEE_STEPS = %d -- nothing to run' % (_done, EMCEE_STEPS))
_pbar = tqdm(initial=_done, total=EMCEE_STEPS)
for _ in (_sampler.sample(_p0, iterations=_todo, progress=False) if _todo else []):
    _pbar.update(1)
    if _sampler.iteration % 100:
        continue
    try:
        _tau = _sampler.get_autocorr_time(tol=0)
        _msg = 'tau %.0f-%.0f' % (np.nanmin(_tau), np.nanmax(_tau))
    except Exception:
        _msg = 'tau not yet estimable'
    tqdm.write('  step %5d | acceptance %.3f | logP max %.2f | %s'
               % (_sampler.iteration, np.mean(_sampler.acceptance_fraction),
                  np.max(_sampler.get_log_prob()[-1]), _msg))
_pbar.close()
print('done in %.0f s' % (time.time() - t0))

# ---- results ------------------------------------------------------------------------------
_tau = _sampler.get_autocorr_time(tol=0)
_tau_ok = bool(np.all(np.isfinite(_tau)))
BURNIN = int(3 * np.nanmax(_tau)) if _tau_ok else max(1, _sampler.iteration // 5)
BURNIN = min(BURNIN, _sampler.iteration - 1)
_thin  = max(1, int(0.5 * np.nanmin(_tau))) if _tau_ok else 1
chain   = _sampler.get_chain()                 # (steps, walkers, NDIM)
logprob = _sampler.get_log_prob()
flat_samples = _sampler.get_chain(discard=BURNIN, thin=_thin, flat=True)
print('\nautocorrelation time %.0f - %.0f steps | burn-in %d, thin %d -> %d samples | acceptance %.3f'
      % (np.nanmin(_tau), np.nanmax(_tau), BURNIN, _thin, len(flat_samples),
         np.mean(_sampler.acceptance_fraction)))
if _sampler.iteration < 50 * np.nanmax(_tau):
    print('WARNING: the chain is shorter than 50 tau, below which emcee calls tau unreliable.')

print('\n%30s %10s %10s %10s %10s %10s %7s' % ('', 'mean', 'std', '2.5%', '97.5%', 'truth', 'pull'))
for i, name in enumerate(param_names):
    s = flat_samples[:, i]
    truth = np.nan if i == J_LAM else float(np.asarray(theta_true)[i])
    pull = (s.mean() - truth) / s.std() if (s.std() > 0 and np.isfinite(truth)) else np.nan
    print('%30s %10.4f %10.4f %10.4f %10.4f %10s %7s'
          % (name, s.mean(), s.std(), np.percentile(s, 2.5), np.percentile(s, 97.5),
             '--' if not np.isfinite(truth) else '%10.4f' % truth,
             '--' if not np.isfinite(pull) else '%7.2f' % pull))
_q = np.percentile(10 ** flat_samples[:, param_names.index('log_Omega')] * KPCGYR_TO_KMS, [16, 50, 84])
print('   Omega_bar = %.2f +%.2f / -%.2f km/s/kpc  (10**log_Omega is in 1/Gyr)'
      % (_q[1], _q[2] - _q[1], _q[1] - _q[0]))

pickle.dump({'chain': chain, 'logprob': logprob, 'flat_samples': flat_samples,
             'param_names': param_names, 'tau': _tau, 'burnin': BURNIN, 'thin': _thin,
             'acceptance': _sampler.acceptance_fraction}, open(OUTPUT_FILE, 'wb'))
print('saved -> %s  (and the live HDF5 backend %s)' % (OUTPUT_FILE, EMCEE_FILE))


In [ ]:
# Corner, with the TRUTH marked -- the point of using a mock.
flat = chain[BURNIN:].reshape(-1, NDIM)
truths = list(np.asarray(theta_true))
truths[J_LAM] = None          # lambda has no truth: it is an estimator setting

fig = corner.corner(flat, labels=param_names, truths=truths, truth_color='tab:red',
                    quantiles=[0.16, 0.5, 0.84], show_titles=True, title_fmt='.3f',
                    title_kwargs={'fontsize': 8}, label_kwargs={'fontsize': 8})
fig.suptitle('AIC pipeline on the mock -- red = the generating truth,  $S^2$ = %.2f'
             % S2_ADOPTED, fontsize=13)
plt.show()

print('\n%-26s %10s %10s %10s %8s' % ('parameter', 'truth', 'median', '+/-', 'pull'))
for j, n in enumerate(param_names):
    q16, q50, q84 = np.percentile(flat[:, j], [16, 50, 84])
    s = 0.5 * (q84 - q16)
    if j == J_LAM:
        print('%-26s %10s %10.4f %10.4f %8s' % (n, '--', q50, s, '--'))
    else:
        t = float(np.asarray(theta_true)[j])
        print('%-26s %10.4f %10.4f %10.4f %8.2f' % (n, t, q50, s, (q50 - t) / s if s > 0 else np.nan))
print('\nTHE TEST: every |pull| should be <~ 2.')
